In [ ]:
from quantile_network_pytorch import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
df_data = pd.read_json("data/ml_data.json").set_index('ievt_n')
df_data = df_data.dropna()
df_data

In [ ]:
xdata = df_data[['E1', 'E2', 'Esum', 'dt1', 'dt2']].to_numpy()
xdata

In [ ]:
ydata = df_data[['Es', 'AoEs']].to_numpy()
ydata

In [ ]:
train_in = xdata
train_out = ydata

In [ ]:
plt.figure()

_ = plt.hist(
    train_in[:, 2], range=(0,4000), bins=400, histtype='step'
)
plt.yscale('log')
plt.xlabel('train_in[2] == Esum (normalised)')


In [ ]:
## Do a test train split
train_in_split, test_in, train_out_split, test_out = train_test_split(
    train_in,
    train_out,
    test_size=1/3,
    random_state=42
)

In [ ]:
len(xdata), len(train_in), len(train_in_split), len(test_in) 

In [ ]:
# normalise output
input_dims  = len(train_in.T)
output_dims = len(train_out.T)

norm_info_in =[[0,1] for i in range(input_dims)]  # µ and stdev
norm_info_out=[[0,1] for i in range(output_dims)] # µ and stdev

train_in_norm = train_in_split.copy()
train_out_norm = train_out_split.copy()
test_in_norm = test_in.copy()
test_out_norm = test_out.copy()

for x in range(input_dims):
    norm_info_in[x] = [np.mean(train_in_split[:,x]), np.std(train_in_split[:,x])]
    train_in_norm[:,x]=(train_in_split[:,x]-np.mean(train_in_split[:,x]))/(np.std(train_in_split[:,x]))
    test_in_norm[:,x]=(test_in[:,x]-np.mean(train_in_split[:,x]))/(np.std(train_in_split[:,x]))

for x in range(output_dims):
    norm_info_out[x] = [np.mean(train_out_split[:,x]), np.std(train_out_split[:,x])]
    train_out_norm[:,x]=(train_out_split[:,x]-np.mean(train_out_split[:,x]))/(np.std(train_out_split[:,x]))
    test_out_norm[:,x]=(test_out[:,x]-np.mean(train_out_split[:,x]))/(np.std(train_out_split[:,x]))

In [ ]:
plt.figure()

_ = plt.hist(
    train_in_norm[:, 2], range=(-2,4), bins=400, histtype='step'
)
plt.yscale('log')


In [ ]:
x_val, y_val = make_dataset(
    train_in_norm,          # input x
    train_out_norm,         # input y
    input_dims,             # x dims
    output_dims,            # y dims
    train_in_norm.shape[0]  # number of samples
) # examples


In [ ]:
# in out case x_val should be twice as long a out training data
len(train_in_norm), len(x_val)

In [ ]:
plt.figure()

_ = plt.hist(
    y_val[:len(train_in_norm)], range=(-4, 4), bins=1000, histtype='step', label='PSS E'
)
_ = plt.hist(
    y_val[len(train_in_norm):], range=(-4, 4), bins=1000, histtype='step', label='A/E'
)
plt.legend()

In [ ]:
width       = 50
hidden      = 5
patience    = 100
cycles      = 4
initiallr   = 0.001
epochs      = 100
batch       = 512
normalise   = True
layer_output_dims = 5 if normalise else 1


In [ ]:
# Define model
if normalise:
    model = QuantileNet(network_type="normalizing")
else:
    model = QuantileNet(network_type="no normalizing")

# Input layer
model.add(nn.Linear(in_features=input_dims + 1, out_features=width))
model.add(nn.LeakyReLU())  # optional: negative_slope=alpha

# Hidden layers
for _ in range(hidden - 1):
    model.add(nn.Linear(in_features=width, out_features=width))
    model.add(nn.LeakyReLU())  # optional: negative_slope=alpha

# Output layer (should be 2 if normalizing, 1 if no normalizing)
model.add(nn.Linear(in_features=width, out_features=layer_output_dims))

In [ ]:
# Store loss history
epoch_list = []
train_loss_list = []
val_loss_list = []


# Create Dataloaders
train_dataset = TensorDataset(iqn_train_in, iqn_train_out)
val_dataset = TensorDataset(iqn_val_in, iqn_val_out)

train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch, shuffle=False)

total_epochs = 0

for x in range(cycles):
    # Adjust learning rate for this cycle
    lr = initiallr * (10 ** -x)

    optimizer = optim.Adam(model.parameters(), lr=lr, amsgrad=False)
    loss_fn = model.loss_fn  # Either normalizing_loss or no_normalizing_loss
    early_stopping = EarlyStopping(patience=patience)

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = loss_fn(batch_y, preds)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        with torch.no_grad():
            val_losses = []
            for val_x, val_y in val_loader:
                val_preds = model(val_x)
                val_loss = loss_fn(val_y, val_preds)
                val_losses.append(val_loss.item())
            mean_val_loss = np.mean(val_losses)

        # Logging
        train_loss_list.append(np.mean(train_losses))
        val_loss_list.append(mean_val_loss)
        epoch_list.append(total_epochs + 1)

        print(f"[Cycle {x+1}, Epoch {epoch+1}] Train: {np.mean(train_losses):.4f}, Val: {mean_val_loss:.4f}")

        early_stopping(mean_val_loss, model)
        if early_stopping.early_stop:
            model.load_state_dict(early_stopping.best_weights)
            break

        total_epochs += 1

# Save the trained model
torch.save(model.state_dict(), 'emulator.pt')

# Final numpy arrays for plotting
train_loss = np.array(train_loss_list)
val_loss = np.array(val_loss_list)
epoch_list = np.array(epoch_list)